In [12]:
from flask import Flask, request, jsonify
import os
import requests
import sys
from flask_apscheduler import APScheduler
from apscheduler.triggers.interval import IntervalTrigger
from datetime import datetime, timedelta
import shutil
import cv2
from ultralytics import YOLO
from tracker import *
import time
import pytz
import pandas as pd

In [14]:
model = YOLO("yolov8n.pt")
cap=cv2.VideoCapture('highway.mp4')
tracker=Tracker()

In [15]:
class_list = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard',
              'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

In [4]:
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (1020, 500))

    results = model.predict(frame)
    a = results[0].boxes.data
    a = a.detach().cpu().numpy()
    px = pd.DataFrame(a).astype("float")
    # print(px)
    list = []
    for index, row in px.iterrows():
        x1 = int(row[0])
        y1 = int(row[1])
        x2 = int(row[2])
        y2 = int(row[3])
        d = int(row[5])
        c = class_list[d]
        if c in ['car', 'bicycle', 'motorcycle', 'bus', 'truck']:
            list.append([x1, y1, x2, y2, c])
    bbox_id = tracker.update(list)

    for bbox in bbox_id:
        x3, y3, x4, y4,vehicle_type, id = bbox
        cx = int(x3 + x4) // 2
        cy = int(y3 + y4) // 2
        cv2.circle(frame,(cx,cy),4,(0,0,255), -1)
        cv2.rectangle(frame,(x3,y3),(x4,y4),(0,0,255),2)
        # cv2.putText(frame, str(id), (x3, y3 - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        # cv2.putText(frame, f"{cx}, {cy}", (x3, y3 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    cv2.imshow("frame", frame)
    if cv2.waitKey(0)&0xFF==27 :
        break
cap.release()
cv2.destroyAllWindows



0: 320x640 6 cars, 1 truck, 176.4ms
Speed: 11.9ms preprocess, 176.4ms inference, 16.3ms postprocess per image at shape (1, 3, 320, 640)


KeyboardInterrupt: 

Vẽ đường tính vận tốc

In [9]:
red_line_y = 198
blue_line_y = 268
text_color = (0, 0, 0)  # Black color for text
yellow_color = (0, 255, 255)  # Yellow color for background
red_color = (0, 0, 255)  # Red color for lines
blue_color = (255, 0, 0)  #

In [40]:
while True:
    ret, frame = cap.read()
    if not ret:
        break
    # count += 1
    # if count % 2 != 0:
    #     continue
    frame = cv2.resize(frame, (1020, 500))

    results = model.predict(frame)
    a = results[0].boxes.data
    a = a.detach().cpu().numpy()
    px = pd.DataFrame(a).astype("float")
    print(px)
    list = []
    for index, row in px.iterrows():
        x1 = int(row[0])
        y1 = int(row[1])
        x2 = int(row[2])
        y2 = int(row[3])
        d = int(row[5])
        c = class_list[d]
        if c in ['car', 'bicycle', 'motorcycle', 'bus', 'truck']:
            list.append([x1, y1, x2, y2, c])
    bbox_id = tracker.update(list)

    for bbox in bbox_id:
        x3, y3, x4, y4,vehicle_type, id = bbox
        cx = int(x3 + x4) // 2
        cy = int(y3 + y4) // 2
        print(bbox)
        print(cx,":",cy)
        # cv2.circle(frame,(cx,cy),4,(0,0,255), -1)
        # cv2.rectangle(frame,(x3,y3),(x4,y4),(0,0,255),2)
    cv2.line(frame, (172, 198), (774, 198), red_color, 2)

    cv2.putText(frame, ('Red Line 198'), (172, 198), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)

    cv2.line(frame, (8, 268), (927, 268), blue_color, 2)

    cv2.putText(frame, ('Blue Line 268'), (8, 268), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)
    cv2.imshow("frame", frame)
    if cv2.waitKey(0)&0xFF==27 :
        break
cap.release()
cv2.destroyAllWindows


0: 320x640 5 cars, 2 trucks, 93.2ms
Speed: 3.5ms preprocess, 93.2ms inference, 2.0ms postprocess per image at shape (1, 3, 320, 640)
            0           1           2           3         4    5
0  578.081055  211.895889  624.110901  248.081024  0.878826  2.0
1  707.178284  195.880508  749.687195  226.568954  0.793927  2.0
2  608.157349  169.397720  643.912048  193.322357  0.771932  2.0
3  192.912811  128.459000  285.882690  209.760681  0.711789  7.0
4  366.685272  139.576828  392.783234  157.322083  0.703356  2.0
5  594.601990   95.763741  618.723389  109.221794  0.417621  2.0
6  417.287201   91.627167  436.176300  109.864784  0.306280  7.0
601 : 229 --- car
728 : 210 --- car
625 : 181 --- car
238 : 168 --- truck
379 : 148 --- car
606 : 102 --- car
426 : 100 --- truck
[578, 211, 624, 248, 'car', 1]
601 : 229
[707, 195, 749, 226, 'car', 2]
728 : 210
[608, 169, 643, 193, 'car', 0]
625 : 181
[192, 128, 285, 209, 'truck', 5]
238 : 168
[366, 139, 392, 157, 'car', 4]
379 : 148
[594, 95, 

KeyboardInterrupt: 

Chia cell và tìm cell trung tâm

In [15]:
import cv2
import numpy as np
import pandas as pd

# Hàm vẽ các cell và tô đỏ viền ô chứa điểm tọa độ trung tâm của vật thể
def draw_cells_and_center(image, grid_size, object_center):
    # Tính toán chiều cao và chiều rộng của mỗi cell
    height, width = image.shape[:2]
    cell_height = height // grid_size[0]
    cell_width = width // grid_size[1]
    
    # Vẽ các cell trên ảnh
    for i in range(grid_size[0]):
        for j in range(grid_size[1]):
            # Vị trí góc trên bên trái và góc dưới bên phải của cell
            start_x = j * cell_width
            start_y = i * cell_height
            end_x = (j + 1) * cell_width
            end_y = (i + 1) * cell_height
            
            # Vẽ hình chữ nhật cho mỗi cell
            cv2.rectangle(image, (start_x, start_y), (end_x, end_y), (0, 255, 0), 2)
    
    # Xác định cell chứa tọa độ trung tâm
    cell_x = object_center[0] // cell_width
    cell_y = object_center[1] // cell_height
    center_start_x = cell_x * cell_width
    center_start_y = cell_y * cell_height
    center_end_x = (cell_x + 1) * cell_width
    center_end_y = (cell_y + 1) * cell_height

    # Tô đỏ viền cell chứa trung tâm
    cv2.rectangle(image, (center_start_x, center_start_y), (center_end_x, center_end_y), (0, 0, 255), 3)
    
    return image

# Đọc ảnh đầu vào
frame = cv2.imread('a2-1-20240116104558.jpg')
results = model.predict(frame)
a = results[0].boxes.data
a = a.detach().cpu().numpy()
px = pd.DataFrame(a).astype("float")
# print(px)
list = []
for index, row in px.iterrows():
    x1 = int(row[0])
    y1 = int(row[1])
    x2 = int(row[2])
    y2 = int(row[3])
    d = int(row[5])
    c = class_list[d]
    # Tính toán trung tâm của bounding box
    object_center = (abs(x1 + x2) // 2, abs(y1 + y2) // 2)
    
    # Vẽ vòng tròn tại tâm
    cv2.circle(frame, object_center, 4, (255, 255, 255), 1)
    cv2.circle(frame, object_center, 8, (255, 255, 255), -1)  # Bán kính lớn hơn và tô kín
    
    # Hiển thị tọa độ trung tâm
    center_text = f"({object_center[0]}, {object_center[1]})"
    cv2.putText(frame, center_text, (object_center[0] + 10, object_center[1] - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    # Vẽ bounding box
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
    # fontScale = 1.0  # Tăng kích thước chữ
    # thickness = 2    # Tăng độ dày nét chữ
    # label = f"car ({x1}, {y1}, {x2}, {y2})"
    # cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, fontScale, (0, 0, 255), thickness)
    # Vẽ lưới và tô đỏ viền ô chứa trung tâm
    # frame = draw_cells_and_center(frame, (26, 26), object_center)

# Lưu ảnh kết quả
cv2.imwrite('output_image.jpg', frame)

# Hiển thị ảnh kết quả
cv2.imshow("frame", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()



0: 512x640 1 truck, 129.6ms
Speed: 3.4ms preprocess, 129.6ms inference, 1.0ms postprocess per image at shape (1, 3, 512, 640)


In [16]:
down = {}
up = {}
counter_down = []
counter_up = []
offset = 6
count = 0
if not os.path.exists('detected_frames'):
    os.makedirs('detected_frames')
while True:
    ret, frame = cap.read()
    if not ret:
        break
    count += 1
    # if count % 2 != 0:
    #     continue
    frame = cv2.resize(frame, (1020, 500))

    results = model.predict(frame)
    a = results[0].boxes.data
    a = a.detach().cpu().numpy()
    px = pd.DataFrame(a).astype("float")
    list = []

    for index, row in px.iterrows():
        x1 = int(row[0])
        y1 = int(row[1])
        x2 = int(row[2])
        y2 = int(row[3])
        d = int(row[5])
        c = class_list[d]
        if 'car' in c:
            list.append([x1, y1, x2, y2,c])
    bbox_id = tracker.update(list)

    for bbox in bbox_id:
        x3, y3, x4, y4,vehicle_type, id = bbox
        cx = int(x3 + x4) // 2
        cy = int(y3 + y4) // 2

        if red_line_y<(cy+offset) and red_line_y > (cy-offset):
           down[id]=time.time()   # current time when vehichle touch the first line
        if id in down:
          
           if blue_line_y<(cy+offset) and blue_line_y > (cy-offset):
             elapsed_time=time.time() - down[id]  # current time when vehicle touch the second line. Also we a re minusing the previous time ( current time of line 1)
             if counter_down.count(id)==0:
                counter_down.append(id)
                distance = 10 # meters 
                a_speed_ms = distance / elapsed_time
                a_speed_kh = a_speed_ms * 3.6  # this will give kilometers per hour for each vehicle. This is the condition for going downside
                cv2.circle(frame,(cx,cy),4,(0,0,255),-1)
                cv2.rectangle(frame, (x3, y3), (x4, y4), (0, 255, 0), 2)  # Draw bounding box
                cv2.putText(frame,str(id),(x3,y3),cv2.FONT_HERSHEY_COMPLEX,0.6,(255,255,255),1)
                cv2.line(frame, (172, 198), (774, 198), red_color, 2)
                cv2.putText(frame, ('Red Line'), (172, 198), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)
                cv2.line(frame, (8, 268), (927, 268), blue_color, 2)
                cv2.putText(frame, ('Blue Line'), (8, 268), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)
                cv2.putText(frame,str(int(a_speed_kh))+'Km/h',(x4,y4 ),cv2.FONT_HERSHEY_COMPLEX,0.8,(0,255,255),2)
                cv2.imshow("frames", frame)
                if cv2.waitKey(0)&0xFF==27:
    #if cv2.waitKey(0) & 0xFF == 27:
                    break
                
        #####going UP blue line#####     
        if blue_line_y<(cy+offset) and blue_line_y > (cy-offset):
           up[id]=time.time()
        if id in up:

           if red_line_y<(cy+offset) and red_line_y > (cy-offset):
             elapsed1_time=time.time() - up[id]
             # formula of speed= distance/time 
             if counter_up.count(id)==0:
                counter_up.append(id)      
                distance1 = 10 # meters  (Distance between the 2 lines is 10 meters )
                a_speed_ms1 = distance1 / elapsed1_time
                a_speed_kh1 = a_speed_ms1 * 3.6
                cv2.circle(frame,(cx,cy),4,(0,0,255),-1)
                cv2.rectangle(frame, (x3, y3), (x4, y4), (0, 255, 0), 2)  # Draw bounding box
                cv2.putText(frame,str(id),(x3,y3),cv2.FONT_HERSHEY_COMPLEX,0.6,(255,255,255),1)
                cv2.line(frame, (172, 198), (774, 198), red_color, 2)
                cv2.putText(frame, ('Red Line'), (172, 198), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)
                cv2.line(frame, (8, 268), (927, 268), blue_color, 2)
                cv2.putText(frame, ('Blue Line'), (8, 268), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)
                cv2.putText(frame,str(int(a_speed_kh1))+'Km/h',(x4,y4),cv2.FONT_HERSHEY_COMPLEX,0.8,(0,255,255),2)
                cv2.imshow("frames", frame)
                if cv2.waitKey(0)&0xFF==27:
                #if cv2.waitKey(0) & 0xFF == 27:
                    break


    
    text_color = (0, 0, 0)  # Black color for text
    yellow_color = (0, 255, 255)  # Yellow color for background
    red_color = (0, 0, 255)  # Red color for lines
    blue_color = (255, 0, 0)  # Blue color for lines

    # cv2.rectangle(frame, (0, 0), (250, 90), yellow_color, -1)

    cv2.line(frame, (172, 198), (774, 198), red_color, 2)
    cv2.putText(frame, ('Red Line'), (172, 198), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)

    cv2.line(frame, (8, 268), (927, 268), blue_color, 2)
    cv2.putText(frame, ('Blue Line'), (8, 268), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)

    # cv2.putText(frame, ('Going Down - ' + str(len(counter_down))), (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)
    # cv2.putText(frame, ('Going Up - ' + str(len(counter_up))), (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)

    # Save frame
    frame_filename = f'detected_frames/frame_{count}.jpg'
    cv2.imwrite(frame_filename, frame)


    # cv2.imshow("frames", frame)
    # if cv2.waitKey(1) & 0xFF == 27:
    # #if cv2.waitKey(0) & 0xFF == 27:
    #     break

cap.release()
cv2.destroyAllWindows()


0: 320x640 6 cars, 1 truck, 68.4ms
Speed: 5.1ms preprocess, 68.4ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 7 cars, 1 truck, 77.5ms
Speed: 2.5ms preprocess, 77.5ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 6 cars, 1 truck, 64.7ms
Speed: 2.9ms preprocess, 64.7ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 6 cars, 1 truck, 61.5ms
Speed: 3.7ms preprocess, 61.5ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 8 cars, 1 truck, 59.8ms
Speed: 2.0ms preprocess, 59.8ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 6 cars, 1 truck, 59.0ms
Speed: 4.1ms preprocess, 59.0ms inference, 0.0ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 5 cars, 2 trucks, 63.5ms
Speed: 3.9ms preprocess, 63.5ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 7 cars, 1 truck, 58.5ms
Speed: 3.9ms preprocess, 58.5ms 

KeyboardInterrupt: 

Crawl data

In [ ]:

output_file = os.path.join(folder_path, f"camera_video_id_{camera[0]}_time_{current_time}.mp4")
video_url = f"{HOST_CAMERA_VIDEO}{camera[1]}" 
# Gửi request đến URL
response = requests.get(video_url, stream=True)
if response.status_code == 200:
# Mở file ở chế độ ghi nhị phân
    with open(output_file, "wb") as file:
        # Ghi từng phần dữ liệu vào file
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)
    print(f"Video tải xuống thành công! Đã lưu tại: {output_file}")
else:
    print(f"Yêu cầu không thành công. Mã lỗi: {response.status_code}")

In [ ]:
model = YOLO("yolov8n.pt")
cap=cv2.VideoCapture('highway.mp4')
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (1020, 500))

    results = model.predict(frame)
    a = results[0].boxes.data
    a = a.detach().cpu().numpy()
    px = pd.DataFrame(a).astype("float")
    list = []
    for index, row in px.iterrows():
        x1 = int(row[0])
        y1 = int(row[1])
        x2 = int(row[2])
        y2 = int(row[3])
        d = int(row[5])
        c = class_list[d]
        if c in ['car', 'bicycle', 'motorcycle', 'bus', 'truck']:
            list.append([x1, y1, x2, y2, c])
        

In [ ]:
def track_vehicle_speed():
    for day_folder in os.listdir(root_folder):
        day_folder_path = os.path.join(root_folder, day_folder)
        if os.path.isdir(day_folder_path):  # Kiểm tra xem có phải thư mục không
            print(f"Processing folder: {day_folder}")
            backup_day_folder_path = os.path.join(backup_video_path, "data_videos", day_folder)
            if not os.path.exists(backup_day_folder_path):
                os.makedirs(backup_day_folder_path)  # Tạo thư mục con tro
            
            # Duyệt qua các video trong thư mục của ngày đó
            for video_file in os.listdir(day_folder_path):
                video_file_path = os.path.join(day_folder_path, video_file)
                if os.path.isfile(video_file_path):  # Kiểm tra xem có phải file không
                    
                    video_url = f"{root_folder}\\{day_folder}\\{video_file}"
                    camera_id = int(video_file.split('_')[3])
                    time_part =  f"{day_folder}:{video_file.split("time_")[1].split(".")[0]}"
                    part = video_file.split("time_")[1].split(".")[0].split("_")
                    print(f" - Video file: {video_url}")
                    cap = cv2.VideoCapture(video_url)
                    if not cap.isOpened():
                        print("Không thể mở video từ URL.")
                        break
                    # countframe = 0
                    while cap.isOpened():
                        ret, frame = cap.read()

                        if not ret:
                            print("Video đã kết thúc hoặc không thể đọc khung hình.")
                            break
                        frame = cv2.resize(frame, (1020, 500))
                       
                        results = model.predict(frame)

                        a = results[0].boxes.data

                        a = a.detach().cpu().numpy()

                        px = pd.DataFrame(a).astype("float")

                        list = []

                        for index, row in px.iterrows():
                            x1 = int(row[0])
                            y1 = int(row[1])
                            x2 = int(row[2])
                            y2 = int(row[3])
                            d = int(row[5])
                            c = class_list[d]
                            # or 'bus' or 'train' or 'truck'
                            if c in ['car', 'bicycle', 'motorcycle', 'bus', 'truck']:
                                list.append([x1, y1, x2, y2, c])
                        bbox_id = tracker.update(list)

                        for bbox in bbox_id:

                            x3, y3, x4, y4, vehicle_type, id = bbox

                            cx = int(x3 + x4) // 2

                            cy = int(y3 + y4) // 2

                            if red_line_y < (cy+offset) and red_line_y > (cy-offset):
                                down[id] = time.time()
                            if id in down:
                                if blue_line_y < (cy+offset) and blue_line_y > (cy-offset):
                                    elapsed_time = time.time() - down[id]
                                    if(elapsed_time == 0):
                                        elapsed_time = 1
                                    if counter_down.count(id) == 0:
                                        counter_down.append(id)
                                        distance = 10  # meters
                                        a_speed_ms = distance / elapsed_time
                                        a_speed_kh = a_speed_ms * 3.6
                                        insert_speed_into_database(
                                            conn, cursor, vehicle_type, a_speed_kh, time_part, camera_id,day_folder,part[0],part[1] )
                            if blue_line_y < (cy+offset) and blue_line_y > (cy-offset):

                                up[id] = time.time()

                            if id in up:

                                if red_line_y < (cy+offset) and red_line_y > (cy-offset):

                                    elapsed1_time = time.time() - up[id]
                                    if(elapsed1_time == 0):
                                        elapsed1_time = 1
                                    # formula of speed= distance/time

                                    if counter_up.count(id) == 0:
                                        counter_up.append(id)
                                        # meters  (Distance between the 2 lines is 10 meters )
                                        distance1 = 5

                                        a_speed_ms1 = distance1 / elapsed1_time

                                        a_speed_kh1 = a_speed_ms1 * 3.6

                                        cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

                                        cv2.rectangle(frame, (x3, y3), (x4, y4),
                                                    (0, 255, 0), 2)  # Draw bounding box

                                        cv2.putText(frame, str(id), (x3, y3),
                                                    cv2.FONT_HERSHEY_COMPLEX, 0.6, (255, 255, 255), 1)

                                        cv2.putText(frame, str(int(a_speed_kh1))+'Km/h', (x4, y4),
                                                    cv2.FONT_HERSHEY_COMPLEX, 0.8, (0, 255, 255), 2)
                                        insert_speed_into_database(
                                            conn, cursor, vehicle_type, a_speed_kh1, time_part, camera_id, day_folder, part[0], part[1] )
                        if cv2.waitKey(1) & 0xFF == ord('q'):

                            break
                    
                    cap.release()
                    try:
                        # Di chuyển video vào thư mục backup
                        shutil.move(video_url, backup_day_folder_path)
                        print(f"Đã di chuyển video {video_file} vào thư mục backup.")
                        
                        # Xóa video trong thư mục gốc (nếu cần)
                        # os.remove(video_url)
                        # print(f"Đã xóa video {video_file} trong thư mục gốc.")
                    except Exception as e:
                        print(f"Không thể di chuyển {video_file}: {e}")

In [ ]:
if red_line_y < (cy+offset) and red_line_y > (cy-offset):
    down[id] = time.time()
if id in down:
    if blue_line_y < (cy+offset) and blue_line_y > (cy-offset):
        elapsed_time = time.time() - down[id]
        if(elapsed_time == 0):
            elapsed_time = 1
        if counter_down.count(id) == 0:
            counter_down.append(id)
            distance = 10  # meters
            a_speed_ms = distance / elapsed_time
            a_speed_kh = a_speed_ms * 3.6

if blue_line_y < (cy+offset) and blue_line_y > (cy-offset):
    up[id] = time.time()
if id in up:
    if red_line_y < (cy+offset) and red_line_y > (cy-offset):
        elapsed1_time = time.time() - up[id]
        if(elapsed1_time == 0):
            elapsed1_time = 1
        if counter_up.count(id) == 0:
            counter_up.append(id)
            distance1 = 5
            a_speed_ms1 = distance1 / elapsed1_time
            a_speed_kh1 = a_speed_ms1 * 3.6
